# Dự đoán sự cố máy bơm trong một giờ tới

Notebook này thực thi trên Linux VM, từ repo root. Dữ liệu mô phỏng đã có, không nằm trong Git. Kernel Jupyter điều khiển các container Feast/Spark đã kiểm chứng để tránh xung đột thư viện.

Mỗi dòng là **một máy tại một thời điểm**. Vì một máy có nhiều snapshot, bảng nhãn dùng `id = SHA256(machine_id|UTC timestamp)` và `label`, không join chỉ bằng machine_id. Feature chỉ dùng thông tin đã đến trước snapshot. Nhãn tương lai chỉ được ghép sau khi lấy feature.

Protocol 60 ngày đã đóng băng trước các lượt train trước: train 42 ngày, validation 9 ngày, test 9 ngày; purge nhãn chưa sẵn sàng tại ranh giới. Notebook này không mở lại test, không đổi protocol sau khi biết kết quả cũ.

In [1]:
from pathlib import Path
import os, json
from ml.notebook_runtime import NotebookRuntime
from ml.protocol import load_protocol, PROTOCOL_SHA256
ROOT = Path.cwd()
protocol = load_protocol(ROOT / 'ml/protocols/medium-60d-v1.json')
runtime = NotebookRuntime(ROOT, os.environ['IOT_NOTEBOOK_RUN'], os.environ['IOT_VM_DEADLINE'])
print(json.dumps({'protocol_sha256': PROTOCOL_SHA256, 'features': protocol['feature_allowlist'],
                  'split_boundaries': protocol['chronological_split'], 'run': runtime.run}, indent=2))

{
  "protocol_sha256": "e0bfcea57331b88bb6914f9342a9477eab6687f6a3929b8cb9834251fa9b8a96",
  "features": [
    "temperature_mean_10m",
    "vibration_max_10m",
    "current_mean_10m",
    "current_std_10m",
    "rpm_mean_10m",
    "reading_count_10m",
    "machine_age_days",
    "prior_failures_24h"
  ],
  "split_boundaries": {
    "start_inclusive": "2026-10-01T00:00:00Z",
    "train_end_exclusive": "2026-11-12T00:00:00Z",
    "validation_end_exclusive": "2026-11-21T00:00:00Z",
    "observation_end_exclusive": "2026-11-30T00:00:00Z",
    "label_available_must_precede_next_split": true
  },
  "run": "notebook-medium-27"
}


## 1. Lấy feature bằng Feast và ghép nhãn

[`ml/notebook_load.py`](../ml/notebook_load.py) xuất từng partition của bảng **đúng hai cột `id,label`**, kèm bảng ánh xạ id → máy/thời điểm riêng. Lệnh native `store.get_historical_features(entity_df=queries, features=service, filter_by_created_timestamp=False).to_df()` chỉ nhận hai cột truy vấn máy/thời điểm, không nhận label/split/episode.

Feature trả về được kiểm từng giá trị với nguồn offline bất biến. Hai phép merge `one_to_one` ghép feature với id rồi với label; số dòng và nhãn phải giữ nguyên. Availability được bảo vệ khi xây nguồn feature, không suy ra chỉ từ PIT join. Hai đồng hồ audit thực và event mô phỏng được ghi riêng.

In [2]:
load_report = runtime.load()
print(json.dumps({k: load_report[k] for k in ['total_rows', 'native_rows_this_invocation', 'labels_schema', 'feast_entity_columns', 'split_counts']}, indent=2))
print(json.dumps(runtime.read('input/merge-sample.json'), indent=2))

load: exit0; 516.763 seconds
{
  "total_rows": 3132184,
  "native_rows_this_invocation": 3132184,
  "labels_schema": [
    "id",
    "label"
  ],
  "feast_entity_columns": [
    "machine_id",
    "event_timestamp"
  ],
  "split_counts": [
    {
      "split": "train",
      "rows": 2195668,
      "positive_rows": 93603
    },
    {
      "split": "validation",
      "rows": 468132,
      "positive_rows": 19828
    },
    {
      "split": "test",
      "rows": 468384,
      "positive_rows": 19957
    }
  ]
}
[
  {
    "id": "984e44253d15839782bbff562e9e347fbe1cd207c2cf5d8e05edc37195e967aa",
    "machine_id": "P-001",
    "event_timestamp": "2026-10-01 02:55:00+00:00",
    "label": 0,
    "temperature_mean_10m": 67.60400000000001,
    "vibration_max_10m": 4.607,
    "current_mean_10m": 21.035857142857143,
    "current_std_10m": 1.9376045172599392,
    "rpm_mean_10m": 3010.057142857143,
    "reading_count_10m": 7,
    "machine_age_days": 180.12152777777777,
    "prior_failures_24h": 0
  }

## 2. Chia tập theo thời gian

Dùng lại hàm `split` trong [`ml/training_job.py`](../ml/training_job.py), cũng là kernel của KFP. Hàm suy lại split từ event time, kiểm `label_available_ts < cutoff`, null/finite, unique key và đủ class/episode rồi ghi ba tập. Không random split vì các snapshot lân cận có thể cùng một sự cố. Việc thống kê metadata test ở đây không dùng để chọn model hay ngưỡng.

In [3]:
partitions = runtime.spark('split')
print(json.dumps(partitions['counts'], indent=2))
assert set(partitions['split_paths']) == {'train', 'validation', 'test'}

split: exit0; 59.129 seconds
{
  "validation": {
    "rows": 468132,
    "positive_rows": 19828,
    "failure_episodes": 1677,
    "first": "2026-11-12T00:00:00",
    "last": "2026-11-20T22:55:00",
    "negative_rows": 448304
  },
  "test": {
    "rows": 468384,
    "positive_rows": 19957,
    "failure_episodes": 1687,
    "first": "2026-11-21T00:00:00",
    "last": "2026-11-29T22:55:00",
    "negative_rows": 448427
  },
  "train": {
    "rows": 2195668,
    "positive_rows": 93603,
    "failure_episodes": 7870,
    "first": "2026-10-01T00:00:00",
    "last": "2026-11-11T22:55:00",
    "negative_rows": 2102065
  }
}


## 3. Huấn luyện và chọn ngưỡng bằng validation

Hàm `train` dựng `Pipeline([VectorAssembler, StandardScaler, LogisticRegression])` rồi gọi `estimator.fit(weighted)`. Chỉ train được dùng để fit scaler và trọng số class nghịch đảo tần suất. `maxIter=40`, `regParam=0.1` giữ nguyên protocol. Đây là binary classification phù hợp nhãn sự cố 0/1.

Validation dùng chọn một ngưỡng trong 0.1..0.9 theo F1, rồi precision, rồi ngưỡng cao hơn khi hòa. Kernel lưu model Spark và JSON portable, seal lựa chọn trước mọi lần mở holdout. Các hàm dùng chung giữ notebook và pipeline nhất quán, không phải một bản thuật toán copy riêng.

In [4]:
selection = runtime.spark('train')
print(json.dumps({'validation': selection['validation'], 'class_weights': selection['train_class_weights'],
                  'iterations': selection['spark_model_iterations'], 'scaler_fit_partition': selection['scaler_fit_partition'],
                  'holdout_rows_opened': selection['holdout_rows_opened'],
                  'saved_spark_model': selection['model_path'], 'saved_portable_model': selection['portable_path']}, indent=2))
assert selection['holdout_rows_opened'] == 0

train: exit0; 68.273 seconds
{
  "validation": {
    "tp": 12748,
    "fp": 15222,
    "fn": 7080,
    "tn": 433082,
    "threshold": 0.7,
    "precision": 0.4557740436181623,
    "recall": 0.6429291910429695,
    "f1": 0.533411439809197,
    "area_under_pr": 0.5957776550421778
  },
  "class_weights": {
    "positive": 11.72861980919415,
    "negative": 0.5222645351119019
  },
  "iterations": 10,
  "scaler_fit_partition": "train",
  "holdout_rows_opened": 0,
  "saved_spark_model": "/ml-data/trained/spark-model",
  "saved_portable_model": "/ml-data/trained/portable.json"
}


## 4. Nạp lại model đã lưu và đánh giá validation

[`ml/notebook_validate.py`](../ml/notebook_validate.py) gọi `PipelineModel.load`, dự đoán lại toàn validation, kiểm confusion counts đúng selection đã seal. Đồng thời so 200 score native Spark với JSON portable. Không đổi ngưỡng, không mở test. PR-AUC trong selection là tích phân hình thang Spark `areaUnderPR`, `numBins=0`, không gọi là average precision.

In [5]:
verification = runtime.validate_saved_model()
print(json.dumps(verification, indent=2))
assert verification['passed'] and verification['holdout_rows_opened'] == 0
assert verification['max_abs_probability_difference'] <= 1e-12

validate: exit0; 39.694 seconds
{
  "passed": true,
  "protocol_sha256": "e0bfcea57331b88bb6914f9342a9477eab6687f6a3929b8cb9834251fa9b8a96",
  "validation": {
    "tp": 12748,
    "fp": 15222,
    "fn": 7080,
    "tn": 433082,
    "threshold": 0.7,
    "precision": 0.4557740436181623,
    "recall": 0.6429291910429695,
    "f1": 0.533411439809197,
    "area_under_pr": 0.5957776550421778
  },
  "restored_validation_confusion": {
    "tp": 12748,
    "fp": 15222,
    "fn": 7080,
    "tn": 433082,
    "threshold": 0.7,
    "precision": 0.4557740436181623,
    "recall": 0.6429291910429695,
    "f1": 0.533411439809197
  },
  "portable_native_probe_rows": 200,
  "max_abs_probability_difference": 2.220446049250313e-16,
  "selection_sha256": "ebaeb08bd56c586fe8cc68a714eab7a52a5f0cdc56c6e3bc584c49a4225a6b65",
  "portable_sha256": "deec94452f7bd419c75cc313ce9cc4e84f5a1049f7d147048958a9607c61e0f3",
  "holdout_rows_opened": 0,
  "code_sha256": "7124f271ff1ae4231ec2265e299d4a946a0edc5c9d658961e6e05a

## Cách trình bày và giới hạn

Tôi có thể giải thích chuỗi: telemetry đã đến → feature → snapshot id → nhãn → temporal split → fit trên train → chọn ngưỡng trên validation → model đã lưu và nạp lại. Risk score chưa calibration, không diễn giải thành xác suất thật của máy bơm. Dữ liệu tổng cộng 3.132.184 snapshot thuộc cohort trung gian 60 ngày, không phải huấn luyện trên nguồn 100 GB.

Notebook dùng Spark `local[2]` và bốn container chạy nối tiếp. Đây chưa là distributed training. Mốc tiếp dùng các executor riêng và giữ nguyên kernels/protocol; model đang phục vụ vẫn là version 1, không tự đổi sang model của notebook.